# CRISP-DM Stage 1: Data Understanding

Notebook ini mencakup tiga aktivitas utama Data Understanding:
1. **Describe Data** (Statistik Deskriptif)
2. **Explore Data** (EDA & Korelasi)
3. **Verify Data Quality** (Kelengkapan, Duplikasi, & Outlier)

In [ ]:
# Parameter Injeksi (DVC / Papermill)
top_provinces_limit = 10
top_regencies_limit = 10


## 1. Describe Data
Menghitung ringkasan statistik deskriptif (*count, mean, median, std dev, min, max*) untuk indikator operasional KDMP.

In [ ]:
import os, json, pandas as pd
from IPython.display import display, Markdown
from config import RAW_REGENCIES_CSV, DATA_DESC_REPORT_MD, DATA_DESC_METRICS_JSON, NUMERIC_COLUMNS

df_reg = pd.read_csv(RAW_REGENCIES_CSV)

# 1. Statistik Deskriptif & Markdown Table
desc_df = df_reg[NUMERIC_COLUMNS].describe(percentiles=[0.5]).T.rename(columns={'50%': 'median', 'std': 'std_dev'})
stats_summary = desc_df[['count', 'mean', 'median', 'std_dev', 'min', 'max']].round(2)
stats_table = stats_summary.to_markdown()

# 2. Simpan Metrik JSON
metrics = {
    "data_description": {
        "total_rows": len(df_reg),
        "total_cols": len(df_reg.columns),
        "variables": {c: {k: round(float(v), 2) for k, v in desc_df.loc[c, ['mean', 'median', 'std_dev', 'min', 'max']].items()} for c in NUMERIC_COLUMNS}
    }
}
os.makedirs(os.path.dirname(DATA_DESC_METRICS_JSON), exist_ok=True)
with open(DATA_DESC_METRICS_JSON, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

# 3. Buat & Simpan Laporan Markdown
report_text = f"""# Laporan Deskripsi Data (CRISP-DM Data Understanding)

- **Total Entitas (Kabupaten/Kota)**: {len(df_reg)}
- **Total Atribut/Kolom**: {len(df_reg.columns)}

## Statistik Deskriptif Indikator KDMP

{stats_table}
"""
with open(DATA_DESC_REPORT_MD, 'w', encoding='utf-8') as f:
    f.write(report_text)

display(Markdown(report_text))


## 2. Explore Data (EDA & Correlations)
Analisis korelasi antar-indikator KDMP serta distribusi data nasional dan wilayah.

In [ ]:
import numpy as np, seaborn as sns, matplotlib.pyplot as plt
from config import RAW_PROVINCES_CSV, DATA_EXPLORATION_REPORT_MD, EDA_METRICS_JSON, FIGURES_DIR

df_prov = pd.read_csv(RAW_PROVINCES_CSV)
os.makedirs(FIGURES_DIR, exist_ok=True)

# 1. Heatmap Korelasi Pearson
corr_matrix = df_reg[NUMERIC_COLUMNS].corr().round(2)
plt.figure(figsize=(8, 6.5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriks Korelasi Pearson Indikator KDMP')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_correlation_matrix.png'))
plt.show()

# 2. Barplot Top Provinsi Jumlah Koperasi
top_prov = df_prov.sort_values(by='total_koperasi', ascending=True).tail(top_provinces_limit)
plt.figure(figsize=(10, 5))
sns.barplot(x='total_koperasi', y='province_name', data=top_prov, hue='province_name', legend=False, palette='Blues_r')
plt.title(f'{top_provinces_limit} Provinsi dengan Jumlah Koperasi Terbanyak di Indonesia')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_top_provinces.png'))
plt.show()

# 3. Barplot Top Kabupaten/Kota Nilai Transaksi
top_reg = df_reg.sort_values(by='nilai_transaksi', ascending=True).tail(top_regencies_limit).copy()
top_reg['nilai_juta'] = top_reg['nilai_transaksi'] / 1e6
plt.figure(figsize=(10, 5))
sns.barplot(x='nilai_juta', y='regency_name', data=top_reg, hue='regency_name', legend=False, palette='viridis')
plt.title(f'{top_regencies_limit} Kabupaten/Kota dengan Nilai Transaksi Tertinggi (Juta Rp)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_top_regencies_transaksi.png'))
plt.show()

# 4. Distribusi Fitur
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
sns.histplot(df_reg['total_koperasi'], kde=True, ax=axes[0, 0], color='skyblue').set_title('Distribusi Total Koperasi')
sns.histplot(np.log1p(df_reg['simpanan_wajib']), kde=True, ax=axes[0, 1], color='salmon').set_title('Distribusi Log Simpanan Wajib')
sns.histplot(np.log1p(df_reg['volume_transaksi']), kde=True, ax=axes[1, 0], color='lightgreen').set_title('Distribusi Log Volume Transaksi')
sns.histplot(np.log1p(df_reg['nilai_transaksi']), kde=True, ax=axes[1, 1], color='plum').set_title('Distribusi Log Nilai Transaksi')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_feature_distributions.png'))
plt.show()

# 5. Simpan Metrik & Generate Report
total_koperasi = int(df_prov['total_koperasi'].sum())
metrics_eda = {
    "summary": {
        "total_provinces": len(df_prov), "total_regencies": len(df_reg),
        "total_koperasi": total_koperasi,
        "koperasi_memiliki_nib": int(df_prov['koperasi_nib'].sum()),
        "pct_nib": round(df_prov['koperasi_nib'].sum() / total_koperasi * 100, 2),
        "koperasi_memiliki_npwp": int(df_prov['koperasi_npwp'].sum()),
        "pct_npwp": round(df_prov['koperasi_npwp'].sum() / total_koperasi * 100, 2),
        "koperasi_telah_rat": int(df_prov['koperasi_rat'].sum()),
        "pct_rat": round(df_prov['koperasi_rat'].sum() / total_koperasi * 100, 2),
        "total_simpanan_pokok": float(df_prov['simpanan_pokok'].sum()),
        "total_simpanan_wajib": float(df_prov['simpanan_wajib'].sum()),
        "total_nilai_transaksi": float(df_prov['nilai_transaksi'].sum())
    }
}
with open(EDA_METRICS_JSON, 'w', encoding='utf-8') as f:
    json.dump(metrics_eda, f, indent=2)

top_prov_sorted = df_prov.sort_values(by='total_koperasi', ascending=False).head(top_provinces_limit)
top_reg_sorted = df_reg.sort_values(by='nilai_transaksi', ascending=False).head(top_regencies_limit)

top_prov_list = "".join([f"{i}. **{r['province_name']}**: {int(r['total_koperasi']):,} Koperasi\n" for i, (_, r) in enumerate(top_prov_sorted.iterrows(), 1)])
top_reg_list = "".join([f"{i}. **{r['regency_name']}**: Rp {float(r['nilai_transaksi']):,.2f}\n" for i, (_, r) in enumerate(top_reg_sorted.iterrows(), 1)])

eda_report_text = f"""# Laporan Eksplorasi Data (CRISP-DM Data Understanding)

## Ringkasan Agregat Nasional
- **Total Kabupaten/Kota**: {len(df_reg)}
- **Total Koperasi Terdata**: {total_koperasi:,} unit
- **Koperasi Memiliki NIB**: {metrics_eda['summary']['koperasi_memiliki_nib']:,} ({metrics_eda['summary']['pct_nib']}%)
- **Koperasi Memiliki NPWP**: {metrics_eda['summary']['koperasi_memiliki_npwp']:,} ({metrics_eda['summary']['pct_npwp']}%)
- **Koperasi Telah RAT**: {metrics_eda['summary']['koperasi_telah_rat']:,} ({metrics_eda['summary']['pct_rat']}%)
- **Akumulasi Simpanan Pokok**: Rp {metrics_eda['summary']['total_simpanan_pokok']:,.2f}
- **Akumulasi Simpanan Wajib**: Rp {metrics_eda['summary']['total_simpanan_wajib']:,.2f}
- **Total Nilai Transaksi**: Rp {metrics_eda['summary']['total_nilai_transaksi']:,.2f}

## Top Wilayah Berdasarkan Kinerja KDMP
### Provinsi dengan Jumlah Koperasi Terbanyak
{top_prov_list}

### Kabupaten/Kota dengan Nilai Transaksi Tertinggi
{top_reg_list}
"""
with open(DATA_EXPLORATION_REPORT_MD, 'w', encoding='utf-8') as f:
    f.write(eda_report_text)

display(Markdown(eda_report_text))


## 3. Verify Data Quality
Verifikasi mutu dataset: kelengkapan, keunikan, konsistensi penulisan nama wilayah, dan deteksi outlier IQR.

In [ ]:
from config import DATA_QUALITY_REPORT_MD, DATA_QUALITY_METRICS_JSON, IGNORED_METADATA_COLUMNS

# 1. Kelengkapan Data
null_df = pd.DataFrame({
    'Jumlah Nilai Kosong (NaN)': df_reg.isnull().sum(),
    'Persentase Kelengkapan (%)': ((len(df_reg) - df_reg.isnull().sum()) / len(df_reg) * 100).round(2)
})
missing_table = null_df.to_markdown()

# 2. IQR Outlier Analysis
num_cols = [c for c in df_reg.columns if c not in IGNORED_METADATA_COLUMNS]
outlier_data = []
outlier_metrics = {}

for col in num_cols:
    s = pd.to_numeric(df_reg[col], errors='coerce').fillna(0)
    q1, q3 = float(s.quantile(0.25)), float(s.quantile(0.75))
    iqr = q3 - q1
    lower, upper = max(0.0, q1 - 1.5 * iqr), q3 + 1.5 * iqr
    out_cnt = int(((s < lower) | (s > upper)).sum())
    out_pct = round(out_cnt / len(df_reg) * 100, 2)
    skew_val = round(float(s.skew()), 2)
    
    outlier_metrics[col] = {"skewness": skew_val, "q1": q1, "q3": q3, "iqr": iqr, "outliers_count": out_cnt, "outliers_pct": out_pct}
    outlier_data.append({
        'Nama Fitur': col, 'Skewness': skew_val, 'Batas Bawah': round(lower, 2),
        'Batas Atas': round(upper, 2), 'Outliers': out_cnt, 'Persentase (%)': out_pct
    })

outlier_table = pd.DataFrame(outlier_data).to_markdown(index=False)

# 3. Simpan Metrik & Generate Report
metrics_qual = {
    "data_quality": {
        "total_samples": len(df_reg),
        "duplicate_rows": int(df_reg.duplicated().sum()),
        "duplicate_keys": int(df_reg.duplicated(subset=['province_id', 'regency_no']).sum()) if 'province_id' in df_reg.columns else 0,
        "is_naming_capital_standard": bool(df_reg['regency_name'].astype(str).str.isupper().all()),
        "feature_quality": outlier_metrics
    }
}
with open(DATA_QUALITY_METRICS_JSON, 'w', encoding='utf-8') as f:
    json.dump(metrics_qual, f, indent=2)

naming_status = "100% Huruf Kapital Sesuai Standar Administrasi" if metrics_qual["data_quality"]["is_naming_capital_standard"] else "Sebagian Perlu Penyeragaman Kapital"

quality_report_text = f"""# Laporan Verifikasi Kualitas Data (CRISP-DM Data Understanding)

- **Total Sampel**: {len(df_reg)}
- **Duplikasi Baris Penuh**: {metrics_qual['data_quality']['duplicate_rows']}
- **Duplikasi Primary Key (province_id, regency_no)**: {metrics_qual['data_quality']['duplicate_keys']}
- **Konsistensi Format Nama**: {naming_status}

## Tabel Kelengkapan Data
{missing_table}

## Tabel Deteksi Pencilan (Metode IQR)
{outlier_table}
"""
with open(DATA_QUALITY_REPORT_MD, 'w', encoding='utf-8') as f:
    f.write(quality_report_text)

display(Markdown(quality_report_text))
